In [1]:
# Montar el google drive

from google.colab import drive
drive.mount('/content/drive')

MessageError: ignored

## Cargar los paquetes a usar

En primer lugar, vamos a importar algunos módulos comunes, asegurarnos de que MatplotLib traza las figuras en línea y preparar una función para guardar las figuras.

In [ ]:
import numpy as np
import pandas as pd
import random

from sklearn import datasets
from scipy.spatial import distance
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer, normalize
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn import metrics
from sklearn.metrics import silhouette_samples, silhouette_score

import matplotlib.pyplot as plt
import matplotlib.cm as cm

from matplotlib.patches import Ellipse, Polygon



### Carguemos nuestro conjunto de datos de viviendas.





*   Esta vez NO voy a hacer una división entrenamiento-prueba.
*   Puede haber razones para seguir haciéndolo, depende.
* Tampoco voy a separar en X e y

In [ ]:
housing_df = pd.read_csv('/content/drive/My Drive/Curso-UTEC-Estudiantes/Sec_7_Clustering/housing.csv', index_col=1)


In [ ]:
housing_df.info()

In [ ]:
housing_df['ExterQual']

In [ ]:
numeric_features     = ['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt',
                        'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                        'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
                        'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
                        'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars',
                        'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
                        'ScreenPorch', 'PoolArea', 'MiscVal', 'YrSold']

numeric_features += ['SalePrice'] # treat this as a regular numeric feature here!!!

In [ ]:
ordinal_features_reg = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
                        'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
ordinal_features_oth = ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                        'Functional',  'Fence']
categorical_features = list(set(housing_df.columns) - set(numeric_features) - set(ordinal_features_reg))

ordering = ['Po', 'Fa', 'TA', 'Gd', 'Ex']

In [ ]:
numeric_preprocessing = make_pipeline(SimpleImputer(strategy='median'),
                                      StandardScaler())
ordinal_preprocessing = make_pipeline(SimpleImputer(strategy='most_frequent'),
                                      OrdinalEncoder(categories=[ordering]*len(ordinal_features_reg)))
categorical_preprocessing = make_pipeline(SimpleImputer(strategy='constant', fill_value="?"),
                                          OneHotEncoder(handle_unknown='ignore', sparse=False))

In [ ]:
preprocessing = ColumnTransformer([
    ('numeric', numeric_preprocessing, numeric_features),
    ('ordinal', ordinal_preprocessing, ordinal_features_reg),
    ('categorical', categorical_preprocessing, categorical_features)
])

In [ ]:
preprocessing.fit(housing_df);

In [ ]:
preprocessing.fit(housing_df);


ohe_columns = list(preprocessing.named_transformers_['categorical'].named_steps['onehotencoder'].get_feature_names_out(categorical_features))
new_columns = numeric_features + ordinal_features_reg + ohe_columns


In [ ]:
housing_df_enc = pd.DataFrame(preprocessing.transform(housing_df), index=housing_df.index, columns=new_columns)
housing_df_enc.shape

## Podemos dividir estas 1460 casas en grupos similares?

Queremos agrupar las observaciones de forma que
* Los ejemplos del mismo grupo sean lo más parecidos posible;
* Los ejemplos de los distintos grupos sean lo más diferentes posible.



In [ ]:
housing_df_enc.head()

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Usamos 2 * log(n_samples) como una regla de dedo para min_samples
min_samples = 2

# Ajustamos de nuevo NearestNeighbors con el nuevo valor de min_samples
nn = NearestNeighbors(n_neighbors=min_samples)
nn.fit(housing_df_enc)

# Calculamos las distancias para el nuevo valor de min_samples
distances, indices = nn.kneighbors(housing_df_enc)

# Ordenamos las distancias y las graficamos
sorted_distances = np.sort(distances[:, -1])
plt.figure(figsize=(10, 6))
plt.plot(sorted_distances)
plt.xlabel('Index')
plt.ylabel('Distance to k-th nearest neighbor')
plt.title(f'k-th Nearest Neighbor Distance (k={min_samples})')
plt.grid(True)
plt.show()


In [ ]:
# Hacemos zoom en la parte inicial de la gráfica para encontrar el "punto de codo" más claramente
plt.figure(figsize=(12, 6))
plt.plot(sorted_distances[:200])  # Mostramos solo los primeros 50 puntos
plt.xlabel('Index')
plt.ylabel('Distance of k-th nearest neighbor')
plt.title('k-th Nearest Neighbor Distance (Zoomed In)')
plt.grid(True)
plt.show()


In [ ]:
min_samples

In [ ]:
from sklearn.cluster import DBSCAN

# Definimos el modelo DBSCAN con los parámetros elegidos
dbscan = DBSCAN(eps=7.5, min_samples=2)

# Ajustamos el modelo a los datos preprocesados
clusters = dbscan.fit_predict(housing_df_enc)

# Veamos cuántos clusters se han formado y cuántos puntos se consideran ruido (label = -1)
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)

(n_clusters, n_noise)


In [ ]:
import numpy as np

# Definimos un rango de valores para eps y min_samples
eps_values = np.arange(3.0, 5.0, 0.2)  # Aumentamos eps desde 3.0 hasta 5.0
min_samples_values = range(2, 6)  # Probamos valores de min_samples desde 2 hasta 5

# Almacenamos los resultados en un diccionario para posterior visualización
results = {}

# Iteramos sobre los valores de eps y min_samples para calcular el silhouette score
for eps in eps_values:
    for min_samples in min_samples_values:
        # Aplicamos DBSCAN con los parámetros actuales
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        clusters = dbscan.fit_predict(housing_df_enc)

        # Si hay al menos 1 cluster y no todos los puntos son ruido, calculamos el silhouette score
        if len(set(clusters)) > 1 and np.any(clusters != -1):
            silhouette_score_value = silhouette_score(housing_df_enc[clusters != -1], clusters[clusters != -1])
            results[(eps, min_samples)] = silhouette_score_value

# Convertimos los resultados a un DataFrame para facilitar la visualización
results_df = pd.DataFrame(list(results.items()), columns=['Parameters', 'Silhouette Score'])

# Ordenamos los resultados por el mejor silhouette score
results_df_sorted = results_df.sort_values(by='Silhouette Score', ascending=False)

# Mostramos los mejores 5 resultados
results_df_sorted.head(5)


In [ ]:
from sklearn.manifold import TSNE

# Definimos el modelo t-SNE para la reducción de dimensión
tsne = TSNE(n_components=2, perplexity=20, n_iter=1000, random_state=42)

# Aplicamos t-SNE a los datos preprocesados - esto puede tardar un poco debido a la naturaleza del algoritmo
data_tsne = tsne.fit_transform(housing_df_enc)

# Ahora graficamos los resultados
plt.figure(figsize=(10, 8))

# Puntos que no son ruido
mask_not_noise = clusters != -1
plt.scatter(data_tsne[mask_not_noise, 0], data_tsne[mask_not_noise, 1], c=clusters[mask_not_noise],
            cmap='viridis', label='Clustered', alpha=0.5)

# Puntos de ruido
plt.scatter(data_tsne[~mask_not_noise, 0], data_tsne[~mask_not_noise, 1], c='red',
            label='Noise', alpha=0.5)

plt.title('t-SNE visualization of the clusters')
plt.xlabel('t-SNE feature 1')
plt.ylabel('t-SNE feature 2')
plt.legend()
plt.show()


In [ ]:
from sklearn.decomposition import PCA

# Aplicamos PCA con dos componentes para una visualización 2D
pca = PCA(n_components=2)
pca_results = pca.fit_transform(housing_df_enc)

# Creamos un DataFrame con los resultados de PCA y las etiquetas de los clusters
pca_df = pd.DataFrame({
    'PCA1': pca_results[:, 0],
    'PCA2': pca_results[:, 1],
    'Cluster': clusters
})

# Visualizamos los resultados con un scatter plot, usando colores diferentes para cada cluster y el ruido
plt.figure(figsize=(12, 8))
scatter = plt.scatter(pca_df['PCA1'], pca_df['PCA2'], c=pca_df['Cluster'], cmap='viridis', alpha=0.5)
plt.title('Clusters visualized with PCA')
plt.xlabel('PCA Feature 1')
plt.ylabel('PCA Feature 2')

# Añadimos una leyenda para los clusters
plt.legend(*scatter.legend_elements(), title="Clusters")

plt.show()
